## MGS 10-Year Yield: Data Coverage & FOMC-Window Diagnostic

**Purpose:** This notebook diagnoses the reliability of the daily 10-year MGS yield series pulled from BNM's `gov-sec-yield` API before the series is used as the local-rates leg of the Sequencing Around FOMC study. It checks how much of the 2015–2026 sample is missing, whether the gaps are explained by Malaysian public holidays, and most importantly, whether any unexplained gaps fall inside the ±2-day windows around FOMC meeting dates the sequencing analysis relies on.

In [1]:
import sys
import json
import holidays
import pandas as pd
from datetime import datetime
sys.path.append('..')
from modules.source import RAW_DATA_PATH, PROCESSED_DATA_PATH, Sources

START = datetime(2015, 1, 1)
END = datetime(2026, 8, 25)

df = pd.read_csv(f"{RAW_DATA_PATH}/{Sources.MGS_10Y}.csv")
df = df.set_index('date')
df.index = pd.to_datetime(df.index)
mgs_10y_daily = df.copy()

with open('../modules/fomc_dates.json') as f:
   fomc_meta = json.load(f)
   
fomc_dates = [datetime.strptime(d, '%Y-%m-%d') for d in fomc_meta['dates']]
fomc_dates = pd.Series(fomc_dates)
print(f"{len(fomc_dates)} FOMC dates loaded (source last verified {fomc_meta['last_verified']})")

94 FOMC dates loaded (source last verified 2026-08-25)


## 1. Coverage Check

How many business days across the full sample period actually have a 10Y
entry, and how many of it are unexplained by Malaysian holidays.

In [2]:
full_bdays = pd.bdate_range(mgs_10y_daily.index.min(), mgs_10y_daily.index.max())
my_holidays = set(pd.to_datetime(d) for d in holidays.Malaysia(years=range(START.year, END.year + 1)))
no_data_dates = full_bdays.difference(mgs_10y_daily.index)
unexplained = sorted([d for d in no_data_dates if d not in my_holidays])

coverage_rows = []
for year, group in pd.Series(full_bdays).groupby(pd.Series(full_bdays).dt.year):
   missing_in_group = group[~group.isin(mgs_10y_daily.index)]
   
   no_data_days = len(missing_in_group)
   unexplained_days = (~missing_in_group.isin(my_holidays)).sum()
   
   coverage_rows.append({
      "year": year, 
      "business_days": len(group), 
      "no_data_days": no_data_days,
      "unexplained": unexplained_days
   })
coverage_df = pd.DataFrame(coverage_rows)

total_row = {
   "year": "Total",
   "business_days": coverage_df["business_days"].sum(),
   "no_data_days": coverage_df["no_data_days"].sum(),
   "unexplained": coverage_df["unexplained"].sum(),
}
coverage_df.loc[len(coverage_df)] = total_row

print(coverage_df.to_string(index=False))

 year  business_days  no_data_days  unexplained
 2015            260            19            8
 2016            261            23           13
 2017            260            43           31
 2018            261            34           22
 2019            261            29           18
 2020            262            15            3
 2021            261            23           13
 2022            260            25           11
 2023            260            19            7
 2024            262            15            6
 2025            261            28           15
 2026            168            13            4
Total           3037           286          151


## 3. FOMC Overlap Check

The check that actually matters: do any of the unexplained gaps fall inside
a FOMC ±2-day window.

In [26]:
window_dates = set(pd.concat([fomc_dates + pd.Timedelta(days=i) for i in range(-2, 3)]))
unexplained_in_fomc_window = sorted([d for d in unexplained if d in window_dates])

print(f"{len(unexplained_in_fomc_window)} of {len(unexplained)} unexplained gaps fall within a FOMC ±2-day window.")

unexplained_set = set(unexplained)
affected_fomc_dates = pd.Series([
   fd for fd in fomc_dates
   if any((fd + pd.Timedelta(days=i)) in unexplained_set for i in range(-2, 3))
])

print(f"{len(affected_fomc_dates)} distinct FOMC events affected:")
for fd in affected_fomc_dates:
   print(f"  {fd.date()}")

28 of 151 unexplained gaps fall within a FOMC ±2-day window.
22 distinct FOMC events affected:
  2016-01-27
  2016-07-27
  2016-09-21
  2017-02-01
  2017-06-14
  2017-09-20
  2017-12-13
  2018-01-31
  2018-06-13
  2018-11-08
  2018-12-19
  2019-01-30
  2019-03-20
  2019-10-30
  2021-01-27
  2021-04-28
  2021-11-03
  2022-11-02
  2023-02-01
  2024-01-31
  2025-03-19
  2025-12-10


## 4. Missing Data Inspection

For each affected FOMC event window, check which exact days does the missing data days land on.

In [4]:
GREEN_O = "\033[92mO\033[0m"
RED_X = "\033[91mX\033[0m"

def build_visual_grid(mgs_df, fomc_dates_list, window=2):
   offsets = list(range(-window, window + 1))
   rows = []
   for fd in fomc_dates_list:
      row = {"fomc_date": fd.date()}
      for offset in offsets:
         target_date = fd + pd.Timedelta(days=offset)
         if target_date in mgs_df.index and pd.notna(mgs_df.loc[target_date, "yield_close"]):
            row[offset] = GREEN_O
         else:
            row[offset] = RED_X
      rows.append(row)
   return pd.DataFrame(rows).set_index("fomc_date")

grid = build_visual_grid(mgs_10y_daily, affected_fomc_dates)
grid.index.name = None

print("Data availability: -2 to +2 days from affected FOMC dates")
print(grid.to_string(header=False))

Data availability: -2 to +2 days from affected FOMC dates
2016-01-27  X  O  O  O  O
2016-07-27  O  O  X  O  O
2016-09-21  X  O  O  O  O
2017-02-01  X  O  X  O  O
2017-06-14  X  X  X  O  X
2017-09-20  O  X  O  O  X
2017-12-13  O  O  O  O  X
2018-01-31  O  O  X  X  O
2018-06-13  X  O  O  O  X
2018-11-08  X  O  O  O  X
2018-12-19  O  O  O  X  O
2019-01-30  O  O  O  O  X
2019-03-20  O  X  X  O  O
2019-10-30  X  O  O  O  O
2021-01-27  O  O  O  X  O
2021-04-28  O  O  O  X  O
2021-11-03  O  O  X  X  O
2022-11-02  O  O  O  O  X
2023-02-01  O  O  X  O  O
2024-01-31  O  O  O  X  O
2025-03-19  O  X  O  O  O
2025-12-10  X  O  O  O  O


**Finding:** 21 of 22 affected events resolve with a maximum 2-day gap on
either side. One event, **2017-06-14**, has no valid observation within 4
days beforehand - only 1 day after. That event is excluded rather than
forward-filled (see Section 7).

## 5. Summary: Original vs Forward-Fill (2-day cap) Comparison

In [30]:
def get_window_dates(offset_days: int):
   return set(pd.concat([fomc_dates + pd.Timedelta(days=i) for i in range(-offset_days, offset_days + 1)]))

unexplained_fomc_2 = sorted([d for d in unexplained if d in get_window_dates(2)])
unexplained_fomc_1 = sorted([d for d in unexplained if d in get_window_dates(1)])
original_dates = set(mgs_10y_daily.index)

mgs_10y_daily_filled = mgs_10y_daily.reindex(full_bdays)
mgs_10y_daily_filled = mgs_10y_daily_filled.ffill(limit=2)
is_missing = mgs_10y_daily_filled["yield_close"].isna()
unexplained_mask = is_missing & ~mgs_10y_daily_filled.index.isin(my_holidays)
fomc_window_mask = unexplained_mask & mgs_10y_daily_filled.index.isin(get_window_dates(2))

no_data_dates_after = mgs_10y_daily_filled.index[is_missing]
len_unexplained_after = unexplained_mask.sum()
len_fomc_window_after_2 = fomc_window_mask.sum()
len_fomc_window_after_1 = (unexplained_mask & mgs_10y_daily_filled.index.isin(get_window_dates(1))).sum()

data = {
   "original": {
      "no-data dates": len(no_data_dates),
      "unexplained": len(unexplained),
      "±2 days in FOMC window": len(unexplained_fomc_2),
      "±1 day in FOMC window": len(unexplained_fomc_1),
   },
   "after ffill": {
      "no-data dates": len(no_data_dates_after),
      "unexplained": len_unexplained_after,
      "±2 days in FOMC window": len_fomc_window_after_2,
      "±1 day in FOMC window": len_fomc_window_after_2,
   }
}
data = pd.DataFrame(data)
print(data.to_string())
fomc_window_dates_after = mgs_10y_daily_filled.index[fomc_window_mask]

print("\n--- Remaining Unexplained Dates in FOMC Window after Forward Filling ---")
for date in fomc_window_dates_after:
   print(date.strftime("%Y-%m-%d"))

                        original  after ffill
no-data dates                286            8
unexplained                  151            4
±2 days in FOMC window        28            1
±1 day in FOMC window         17            1

--- Remaining Unexplained Dates in FOMC Window after Forward Filling ---
2017-06-14


## 7. Conclusion

A forward-fill capped at 2 days resolves 21 of the 22 affected FOMC windows, cutting unexplained business days to 4 overall and FOMC-window gaps to 1. The one exception is 2017-06-14: no valid print exists in the 4 days before it, so a forward-only fill has nothing to draw on. That event is dropped from the study; every other in-sample FOMC date is either directly observed or bridged by a short, transparent fill.